In [ ]:
import torch

print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
print(f"Device name: {torch.cuda.get_device_name(0)}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import List, Dict

In [ ]:
df_raw = pd.read_csv("data/paysim_dataset.csv")
df_raw.drop(columns=["nameOrig", "nameDest"], inplace=True)

In [ ]:
df_raw.head()

In [ ]:
df_raw_train, df_raw_eval_test = train_test_split(
    df_raw, test_size=0.2, random_state=42
)
df_raw_eval, df_raw_test = train_test_split(
    df_raw_eval_test, test_size=0.5, random_state=42
)

In [ ]:
labels_train = df_raw_train["isFraud"].to_numpy()
df_raw_train.drop(columns=["isFraud", "isFlaggedFraud"], inplace=True)

labels_eval = df_raw_eval["isFraud"].to_numpy()
baselines_eval = df_raw_eval["isFlaggedFraud"].to_numpy()
df_raw_eval.drop(columns=["isFraud", "isFlaggedFraud"], inplace=True)

labels_test = df_raw_test["isFraud"].to_numpy()
baselines_test = df_raw_test["isFlaggedFraud"].to_numpy()
df_raw_test.drop(columns=["isFraud", "isFlaggedFraud"], inplace=True)

In [ ]:
def convert_step_to_hour(step):
    hour = step % 24
    return f"SYNTH_HOUR_{hour:02d}"


def convert_step_to_part_of_day(step):
    hour = step % 24
    if 0 <= hour < 6:
        return "pseudo_night"
    elif 6 <= hour < 12:
        return "pseudo_morning"
    elif 12 <= hour < 18:
        return "pseudo_afternoon"
    else:
        return "pseudo_evening"

In [ ]:
df_raw_train["hour_of_transaction"] = df_raw_train["step"].apply(convert_step_to_hour)
df_raw_eval["hour_of_transaction"] = df_raw_eval["step"].apply(convert_step_to_hour)
df_raw_test["hour_of_transaction"] = df_raw_test["step"].apply(convert_step_to_hour)

df_raw_train.drop(columns=["step"], inplace=True)
df_raw_eval.drop(columns=["step"], inplace=True)
df_raw_test.drop(columns=["step"], inplace=True)

In [ ]:
VALUE_BINS = [
    -np.inf,
    0,
    10_000,
    50_000,
    100_000,
    250_000,
    500_000,
    750_000,
    1_000_000,
    3_000_000,
    np.inf,
]
VALUE_LABELS = [
    "VALUE_EQUAL_0",
    "VALUE_BETWEEN_0_10K",
    "VALUE_BETWEEN_10K_50K",
    "VALUE_BETWEEN_50K_100K",
    "VALUE_BETWEEN_100K_250K",
    "VALUE_BETWEEN_250K_500K",
    "VALUE_BETWEEN_500K_750K",
    "VALUE_BETWEEN_750K_1M",
    "VALUE_BETWEEN_1M_3M",
    "VALUE_ABOVE_3M",
]


def bin_values_column(series):
    """Fast vectorized binning using pd.cut for whole Pandas Series."""
    return pd.cut(series, bins=VALUE_BINS, labels=VALUE_LABELS, right=True)


monetary_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

# Apply binning across columns
binned_df = df_raw[monetary_cols].apply(bin_values_column)

# Plot categorical count plots
fig, axes = plt.subplots(len(monetary_cols), 1, figsize=(10, 12), sharex=True)

for i, col in enumerate(monetary_cols):
    counts = binned_df[col].value_counts().reindex(VALUE_LABELS, fill_value=0)
    sns.barplot(x=counts.index, y=counts.values, ax=axes[i], color="skyblue")
    axes[i].set_title(f"Bin Distribution for {col}")
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
num_cols_train = df_raw_train.select_dtypes(include=["number"]).columns

for col in num_cols_train:
    df_raw_train[col] = bin_values_column(df_raw_train[col])

num_cols_eval = df_raw_eval.select_dtypes(include=["number"]).columns

for col in num_cols_eval:
    df_raw_eval[col] = bin_values_column(df_raw_eval[col])

num_cols_test = df_raw_test.select_dtypes(include=["number"]).columns

for col in num_cols_test:
    df_raw_test[col] = bin_values_column(df_raw_test[col])

In [ ]:
df_raw_train.reset_index(drop=True, inplace=True)
df_raw_eval.reset_index(drop=True, inplace=True)
df_raw_test.reset_index(drop=True, inplace=True)

In [ ]:
special_tokens = {"[CLS]": 0, "[MASK]": 1, "[SEP]": 2}

transaction_types_tokens = df_raw_train["type"].unique().tolist()
value_bins_tokens = VALUE_LABELS
hour_of_transaction_tokens = df_raw_train["hour_of_transaction"].unique().tolist()

values_vocab = transaction_types_tokens + value_bins_tokens + hour_of_transaction_tokens

print("Number of Special Tokens:", len(special_tokens))
print("Number of Transaction Type Tokens:", len(transaction_types_tokens))
print("Number of Value Bin Tokens:", len(value_bins_tokens))
print("Number of Hour of Transaction Tokens:", len(hour_of_transaction_tokens))

vocab_size = (
    len(special_tokens)
    + len(transaction_types_tokens)
    + len(value_bins_tokens)
    + len(hour_of_transaction_tokens)
)

print("\nTotal Vocabulary Size:", vocab_size)

In [ ]:
values2tokens = {
    value: idx + len(special_tokens) for idx, value in enumerate(values_vocab)
}
token2values = {v: k for k, v in values2tokens.items()}

In [ ]:
class FraudMLMDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

        self.cls_id = special_tokens["[CLS]"]
        self.sep_id = special_tokens["[SEP]"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        raw_tokens = self.data.iloc[idx].values.tolist()

        tokens_ids = [values2tokens[token] for token in raw_tokens]

        input_ids = [self.cls_id] + tokens_ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

In [ ]:
train_dataset = FraudMLMDataset(df_raw_train)
eval_dataset = FraudMLMDataset(df_raw_eval)
test_dataset = FraudMLMDataset(df_raw_test)

In [ ]:
@dataclass
class FraudDataCollatorForMLM:
    mask_token_id: int
    cls_token_id: int
    sep_token_id: int
    vocab_size: int
    mlm_probability: float = 0.15
    num_special_tokens: int = 3  # [CLS], [SEP], [MASK]

    def __call__(
        self, examples: List[Dict[str, torch.Tensor]]
    ) -> Dict[str, torch.Tensor]:
        input_ids = torch.stack([e["input_ids"] for e in examples])
        attention_mask = torch.stack([e["attention_mask"] for e in examples])

        labels = input_ids.clone()

        # Mask candidates: not CLS, SEP
        probability_matrix = torch.full(labels.shape, self.mlm_probability)

        special_mask = (input_ids == self.cls_token_id) | (
            input_ids == self.sep_token_id
        )

        probability_matrix.masked_fill_(special_mask, value=0.0)

        masked_indices = torch.bernoulli(probability_matrix).bool()

        # Labels only for masked positions
        labels[~masked_indices] = -100

        # 80% -> [MASK]
        indices_replaced = (
            torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
        )
        input_ids[indices_replaced] = self.mask_token_id

        # 10% -> random token
        indices_random = (
            torch.bernoulli(torch.full(labels.shape, 0.5)).bool()
            & masked_indices
            & ~indices_replaced
        )

        # Random value tokens only, avoid special tokens
        random_words = torch.randint(
            low=self.num_special_tokens,
            high=self.vocab_size,
            size=labels.shape,
            dtype=torch.long,
        )
        input_ids[indices_random] = random_words[indices_random]

        # remaining 10% stay unchanged

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

In [ ]:
collator = FraudDataCollatorForMLM(
    mask_token_id=special_tokens["[MASK]"],
    cls_token_id=special_tokens["[CLS]"],
    sep_token_id=special_tokens["[SEP]"],
    vocab_size=vocab_size,
    mlm_probability=0.15,
    num_special_tokens=len(special_tokens),
)

In [ ]:
from transformers import BertConfig, BertForMaskedLM

config = BertConfig(
    vocab_size=vocab_size,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=df_raw_train.shape[-1] + 2,  # [CLS] and [SEP] tokens
    type_vocab_size=1,
    pad_token_id=None,  # No padding token in this dataset since all sequences are of the same length
)

model = BertForMaskedLM(config)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./fraud_bert_mlm",
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    weight_decay=0.01,
    warmup_steps=100,
    report_to="none",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./fraud_bert_mlm_best")

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

In [ ]:
# Path to your best MLM checkpoint directory
mlm_checkpoint_path = "./fraud_bert_mlm_best"

# Load model weights into a Sequence Classification architecture
model = AutoModelForSequenceClassification.from_pretrained(
    mlm_checkpoint_path,
    num_labels=2,  # Binary classification (0: normal, 1: fraud)
)

In [ ]:
class FraudClassificationDataset(Dataset):

    def __init__(self, dataframe, labels):
        self.data = dataframe.reset_index(drop=True)
        self.labels = torch.tensor(labels, dtype=torch.long)

        self.cls_id = special_tokens["[CLS]"]
        self.sep_id = special_tokens["[SEP]"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        raw_tokens = self.data.iloc[idx].values.tolist()

        # Direct dictionary lookup (no fallback needed)
        tokens_ids = [values2tokens[token] for token in raw_tokens]

        input_ids = [self.cls_id] + tokens_ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": self.labels[idx],
        }

In [ ]:
supervised_train_dataset = FraudClassificationDataset(df_raw_train, labels_train)
supervised_eval_dataset = FraudClassificationDataset(df_raw_eval, labels_eval)
supervised_test_dataset = FraudClassificationDataset(df_raw_test, labels_test)

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_recall_fscore_support,
    average_precision_score,
    roc_auc_score,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Probability of fraud
    probabilities = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

    # Default threshold
    predictions = (probabilities >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0,
    )

    pr_auc = average_precision_score(labels, probabilities)
    roc_auc = roc_auc_score(labels, probabilities)

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./fraud_bert_classification",
    eval_strategy="steps",
    save_strategy="steps",
    learning_rate=2e-5,  # Lower learning rate for fine-tuning
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    metric_for_best_model="pr_auc",  # Ideal metric for imbalanced fraud datasets
    greater_is_better=True,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=supervised_train_dataset,  # Dataset yielding 'input_ids', 'attention_mask', and 'labels'
    eval_dataset=supervised_eval_dataset,
    compute_metrics=compute_metrics,  # Custom function for Precision, Recall, F1
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./fraud_bert_classification_best")

In [ ]:
from sklearn.metrics import classification_report, f1_score

# 1. Compute overall F1 score (binary default)
f1 = f1_score(labels_test, baselines_test)
print(f"F1 Score: {f1:.4f}\n")

# 2. Print comprehensive stats (Precision, Recall, F1, Support)
print(classification_report(labels_test, baselines_test, digits=4))

In [ ]:
test_results = trainer.predict(supervised_test_dataset)

# Extract raw logits and ground truth labels
logits = test_results.predictions
labels = test_results.label_ids

# Calculate true probabilities for Class 1 (Fraud)
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

# View automated metrics (PR-AUC, ROC-AUC, and 0.5-threshold F1)
print(test_results.metrics)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 50)

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.50, 0.02):
    preds = (probs >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    print(f"{t:<10.2f} | {p:<10.4f} | {r:<10.4f} | {f1:<10.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"\nBest Threshold for F1: {best_threshold:.2f} (Yields F1: {best_f1:.4f})")

In [ ]:
final_preds = (probs >= best_threshold).astype(int)

# Full detailed output
print(classification_report(labels, final_preds, digits=4))